# 01 Bronze and silver

**Source:** a scraped Vogue Runway dataset from the Internet Archive.
- `VogueRunway.parquet`: metadata for 1,281,633 images (designer, season, year, section, tags, image URL)
- `VogueRunway_image.npy`: pre-computed CLIP ViT-B/32 embeddings, 512 numbers per image, in the same row order

**This notebook:**
1. Loads the raw metadata into bronze, unchanged except for a row_id and load details
2. Explores the columns to decide cleaning rules
3. Builds a clean silver table

### Load raw metadata into bronze
Reads the raw parquet file and saves it as a Delta table, unchanged except for three columns:
- `row_id`: the key as a number, matching each row's position in the embeddings file
- `_source_file` and `_ingested_at`: where each row came from and when it was loaded

**Check:** the row count should be 1,281,633, the same as the embeddings file.

In [0]:
from pyspark.sql import functions as F

RAW_PATH = "/Volumes/workspace/default/vogue/VogueRunway.parquet"

bronze = (spark.read.parquet(RAW_PATH)
    .withColumn("row_id", F.col("key").cast("int"))
    .withColumn("_source_file", F.lit(RAW_PATH))
    .withColumn("_ingested_at", F.current_timestamp()))

bronze.write.mode("overwrite").saveAsTable("workspace.silhouette_archive.bronze_runway")
print(spark.table("workspace.silhouette_archive.bronze_runway").count())

### Explore section values
Counts images per section to decide which sections belong in silver.

In [0]:
bronze = spark.table("workspace.silhouette_archive.bronze_runway")
display(bronze.groupBy("section").count().orderBy("count", ascending=False))

### Sample images per section
Shows 5 random images from each section, to decide which sections belong in silver.

In [0]:
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
from pyspark.sql.window import Window

# Pick 5 random images per section (seed 42 so it's repeatable)
w = Window.partitionBy("section").orderBy(F.rand(42))
samples = (bronze.withColumn("rn", F.row_number().over(w))
    .filter("rn <= 5")
    .select("section", "designer", "year", "url")
    .collect())

# Draw a grid: one row per section, 5 images across
sections = sorted(set(r.section for r in samples))
fig, axes = plt.subplots(len(sections), 5, figsize=(15, 3.5 * len(sections)))
headers = {"User-Agent": "Mozilla/5.0"}

for i, s in enumerate(sections):
    rows = [r for r in samples if r.section == s]
    for j in range(5):
        ax = axes[i][j]
        ax.axis("off")
        if j < len(rows):
            r = rows[j]
            try:
                img = Image.open(BytesIO(requests.get(r.url, headers=headers, timeout=10).content)).convert("RGB")
                ax.imshow(img)
                ax.set_title(f"{s}\n{r.designer} {r.year}", fontsize=8)
            except Exception as e:
                ax.set_title(f"{s}\nFailed: {type(e).__name__}", fontsize=8)

plt.tight_layout()
plt.show()

### Findings: section samples

- **Front Row:** drop. Celebrities in the audience, tagged with the collection but not wearing it. This is label noise and would produce wrong matches.
- **Beauty:** drop. Makeup and hair close-ups, no garments.
- **Atmosphere:** drop. Wide venue shots, garments too small to match.
- **Collection:** keep. Clear, front-on full looks, runway and studio.
- **Details:** keep. Close-ups of single pieces. Risk: crop mismatch with query photos.

**Decision:** keep Collection and Details in silver with a shot_type column. Which to search is decided later by evaluation: test-set photos scored against Collection, Details, and both.

### Data quality checks on bronze
Counts nulls in key columns, duplicate image URLs, and very small images, to set the remaining silver rules.

In [0]:
from pyspark.sql import functions as F

kept = bronze.filter(F.col("section").isin("Collection", "Details"))
print("Rows in kept sections:", kept.count())

display(kept.select(
    F.sum(F.col("designer").isNull().cast("int")).alias("null_designer"),
    F.sum(F.col("year").isNull().cast("int")).alias("null_year"),
    F.sum(F.col("season").isNull().cast("int")).alias("null_season"),
    F.sum(F.col("url").isNull().cast("int")).alias("null_url"),
    F.sum((F.least("width", "height") < 300).cast("int")).alias("tiny_images"),
))

print("Duplicate URLs:", kept.count() - kept.select("url").distinct().count())
display(kept.groupBy("year").count().orderBy("year"))

In [0]:
tiny = kept.filter(F.least("width", "height") < 300)
display(tiny.groupBy("year").count().orderBy("year"))

### Findings: data quality

- **Nulls:** zero in designer, year, season and url. Metadata is complete.
- **Year spread:** heavily skewed to recent years. Before 2000, only a few thousand images per year. From 2000 it climbs to about 100,000 in 2016, then drops to about 50,000 by 2023. 2024 is partial, since that's when the dataset was scraped.
- **Tiny images:** about 72,000 have a shortest side under 300 pixels. Almost all are from 2000 to 2005, with single digits in other years. That's just how big Vogue's photos were back then.

**Decisions:**
- **Year skew** is handled in the subset by capping images per collection, not in silver.
- **Era filters** in the front end let users search older years without them being drowned out.
- **Keep tiny images.** Dropping them would remove most of 2000 to 2005, the core vintage era. Flag them with an `is_low_res` column in silver, and check in evaluation whether they match worse.

### Write silver
Keeps Collection and Details only, removes duplicate image URLs, and adds two columns:
- `shot_type`: full_look for Collection, detail for Details
- `is_low_res`: true when the shortest side is under 300 pixels

In [0]:
silver = (bronze
    .filter(F.col("section").isin("Collection", "Details"))
    .dropDuplicates(["url"])
    .withColumn("shot_type",
        F.when(F.col("section") == "Details", "detail").otherwise("full_look"))
    .withColumn("is_low_res", F.least("width", "height") < 300))

silver.write.mode("overwrite").saveAsTable("workspace.silhouette_archive.silver_runway")

s = spark.table("workspace.silhouette_archive.silver_runway")
print("Silver rows:", s.count())
display(s.groupBy("shot_type", "is_low_res").count())

### Validate silver
Checks the silver table before anything downstream uses it. Each check stops the notebook with an error if it fails:
- Only Collection and Details sections remain
- No duplicate image URLs
- No nulls in designer, year, season or url
- Every row has a shot_type and an is_low_res flag
- Row count is less than bronze and greater than zero

In [0]:
s = spark.table("workspace.silhouette_archive.silver_runway")
b = spark.table("workspace.silhouette_archive.bronze_runway")
n = s.count()

sections = {r.section for r in s.select("section").distinct().collect()}
assert sections <= {"Collection", "Details"}, f"Unexpected sections: {sections}"

assert s.select("url").distinct().count() == n, "Duplicate URLs found"

for c in ["designer", "year", "season", "url", "shot_type", "is_low_res"]:
    nulls = s.filter(F.col(c).isNull()).count()
    assert nulls == 0, f"{nulls} nulls in {c}"

assert 0 < n < b.count(), f"Unexpected row count: {n}"

print(f"All checks passed: {n:,} rows")